# Oracle RDF end-to-end verification

Use this notebook to validate the Oracle path from driver -> RDF utils -> Oracle operations.

How to use:
1. Provide your initialized `driver` in the setup cell.
2. Run cells top to bottom.
3. The suite writes test data with a unique `group_id`, validates inserts, then cleans up.

This notebook expects an `OracleDriver` with RDF mode enabled.

In [ ]:
from datetime import datetime, timezone
from pprint import pprint
from uuid import uuid4

from graphiti_core.driver.driver import GraphProvider
from graphiti_core.driver.oracle.rdf_utils import execute_sparql_update, get_rdf_identifiers_for_executor
from graphiti_core.edges import CommunityEdge, EntityEdge, EpisodicEdge, HasEpisodeEdge, NextEpisodeEdge
from graphiti_core.models.edges.edge_db_queries import (
    get_community_edge_save_query,
    get_entity_edge_save_bulk_query,
    get_entity_edge_save_query,
)
from graphiti_core.models.nodes.node_db_queries import (
    get_community_node_save_query,
    get_entity_node_save_bulk_query,
    get_entity_node_save_query,
)
from graphiti_core.nodes import CommunityNode, EntityNode, EpisodicNode, EpisodeType, SagaNode

In [ ]:
# Provide your initialized OracleDriver here before running the suite.
# Example:
# from graphiti_core.driver.oracle_driver import OracleDriver
# driver = OracleDriver(
#     dsn='...',
#     user='...',
#     password='...',
#     use_rdf=True,
#     rdf_network_owner='RDFUSER',
#     rdf_network_name='NET1',
#     rdf_graph_name='GRAPHITI',
# )

driver = None

In [ ]:
def _require_oracle_rdf_driver(drv) -> None:
    if drv is None:
        raise RuntimeError('Please assign your initialized OracleDriver to `driver` first.')

    if getattr(drv, 'provider', None) != GraphProvider.ORACLE:
        raise RuntimeError(f'Expected Oracle driver, got provider={getattr(drv, "provider", None)}')

    if not getattr(drv, 'rdf_enabled', False):
        raise RuntimeError('This notebook expects RDF mode enabled (`use_rdf=True` or ORACLE_USE_RDF=true).')


def _run_query_builder_checks() -> None:
    entity_node_query = get_entity_node_save_query(GraphProvider.ORACLE, 'Entity:Person')
    assert 'db.create.setNodeVectorProperty' not in entity_node_query

    entity_node_bulk_query = get_entity_node_save_bulk_query(GraphProvider.ORACLE, [])
    assert 'db.create.setNodeVectorProperty' not in entity_node_bulk_query

    community_node_query = get_community_node_save_query(GraphProvider.ORACLE)
    assert 'db.create.setNodeVectorProperty' not in community_node_query

    entity_edge_query = get_entity_edge_save_query(GraphProvider.ORACLE)
    assert 'db.create.setRelationshipVectorProperty' not in entity_edge_query

    entity_edge_bulk_query = get_entity_edge_save_bulk_query(GraphProvider.ORACLE)
    assert 'db.create.setRelationshipVectorProperty' not in entity_edge_bulk_query

    community_edge_query = get_community_edge_save_query(GraphProvider.ORACLE)
    assert 'WHERE node:Entity OR node:Community' in community_edge_query

    print('Query-builder checks passed.')


async def _count_rdf_triples(drv, rdf_table_name: str) -> int:
    rows, _, _ = await drv.execute_query(f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}')
    if len(rows) == 0:
        return 0
    return int(rows[0]['triple_count'])